# 📘 SalesTeam AI — Synthèse Complète du Projet & Comparatif Avant / Après

> Ce notebook retrace l'intégralité du projet **SalesTeam AI** : des premières étapes de préparation des données brutes jusqu'à la résolution des fuites d'information, la mise en place du business re-ranking et l'intégration du LLM Llama 3.3.

---

## 📌 1. Présentation Générale & Première Étapes du Projet

L'objectif de **SalesTeam AI** est de fournir une solution intelligente d'aide à la décision pour les commerciaux sur le terrain. L'application suggère à chaque visite les produits les plus pertinents à commander, avec la quantité idéale et une justification en langage naturel.

### 🛠️ Étapes de Construction Initiales :
1. **Nettoyage des Données Brutes (`src/data/cleaner.py`)** :
   - Extraction et nettoyage des historiques de factures, de lignes de commande et des coordonnées GPS des clients (`LSAT`, `NEWTECH`, `ONETEL`).
   - Gestion des valeurs manquantes, harmonisation des dates et des identifiants clients/articles.
2. **Fusion de la Table Principale (`src/data/loader.py`)** :
   - Assemblage des tables `commandes_clean.csv`, `lignes_clean.csv` et `gps_clean.csv` dans `main_table.csv` (78 530 lignes brutes couvrant 2024 à juin 2026).


## 📊 2. Explication des Groupes de Features (Feature Engineering)

La matrice de caractéristiques (`src/features/feature_engineering.py`) calcule 24 variables réparties en **5 groupes distincts** :

### 1️⃣ Historique de Commandes Client-Produit (Signal Majeur)
- `avg_qty`, `std_qty`, `min_qty`, `max_qty`, `total_qty` : Statistiques de volume d'achat.
- `frequency` : Nombre total de commandes passées pour cet article.
- `last_qty` : Quantité commandée lors de la dernière transaction.
- `recency_days` : Nombre de jours écoulés depuis le dernier achat de ce produit.
- `avg_delay_days` : Délai moyen habituel entre deux réapprovisionnements (en jours).
- `recency_relative` : Ratio $\frac{\text{recency\_days}}{\text{avg\_delay\_days}}$ (indicateur de retard de réassort).
- `trend` : Évolution récente de la demande (+/-%).

### 2️⃣ Saisonnalité
- `avg_seasonal_coef`, `current_month_coef`, `best_month` : Analyse des pics mensuels de vente.

### 3️⃣ Données Géographiques
- `latitude`, `longitude`, `has_gps` : Coordonnées GPS pour le repérage spatial.

### 4️⃣ Profil Produit
- `categorie`, `designation`, `is_bulk_product`, `nb_clients`, `days_since_first_order`, `is_new_product`.

### 5️⃣ Profil Global Client
- `client_total_products`, `client_total_invoices`, `client_avg_basket_size`.


## 🚨 3. Le Défi de l'Overfitting & La Solution Visit-Level

### ❌ Le Problème Initial (Faux 100% d'Accuracy / AUC 0.9999) :
- Dans la première version (`target_builder.py`), un produit était marqué `target=1` s'il avait été acheté au moins une fois dans la vie du client, et `target=0` s'il n'avait *jamais* été acheté.
- **La Fuite** : La variable `frequency` valait toujours `0` pour les négatifs et `>0` pour les positifs. Le modèle se contentait de vérifier `frequency > 0` sans apprendre le rythme de réassort.

### ✅ La Solution "Visit-Level" :
- Pour chaque facture (visite commerciale) à une date $T$, nous évaluons uniquement les produits que le client connaissait déjà **avant $T$** :
  - `target_bought = 1` : Produit commandé sur cette facture.
  - `target_bought = 0` : Produit connu mais ignoré/skippé lors de cette visite.
- **Résultat** : Un jeu de données de **1 222 876 lignes** avec un taux de positifs réaliste de **3.1%** et **aucun moyen de tricher**.

## 📈 4. Résultats d'Entraînement des Modèles IA

### 🔵 Classifieur XGBoost (`train_classifier.py`)
- **Split Temporel Anti-Fuite** : Train $\le$ 2025-12-31 (987k lignes), Val Q1 2026 (111k lignes), Test Q2 2026+ (124k lignes).
- **Hyperparamètres régulés** : `max_depth=5`, `min_child_weight=5`, `n_estimators=200`.
- **ROC-AUC Final sur Test 2026** : **`0.8608`** *(plage cible idéale 0.75 - 0.85/0.86)*.
- **Variables les plus importantes** :
  1. `recency_days` : **54.1%**
  2. `recency_relative` : **15.2%**
  3. `avg_delay_days` : **6.4%**
  4. `best_month` : **6.3%**
  5. `frequency` : **4.9%**

### 🟠 Régresseur XGBoost (`train_regressor.py`)
- Prédit la quantité spécifique à commander en fonction de `last_qty`, `max_qty`, `std_qty`, `avg_qty` (MAE : ~8.2 unités sur les positifs).

## ⚡ 5. Couche de Business Re-Ranking & Intégration LLM

### 1. Formule de Re-Ranking Métier (`recommendation.py`)
$$\text{final\_score} = \text{ml\_score} \times \text{timing\_boost} \times \text{trend\_boost}$$

- **Timing Boost** : Si le client atteint 150% de son cycle d'achat habitude (`recency_relative >= 1.5`), le produit reçoit un **boost de 3.0x** pour remonter en priorité.
- **Dédoublonnage** : Conservation du dernier snapshot par produit pour éliminer les doublons d'articles dans les suggestions.

### 2. Explications LLM (`explanation.py`)
- Connexion à l'Inference API HuggingFace (`meta-llama/Llama-3.3-70B-Instruct:fastest`).
- Cache mémoire 24h et système de repli (fallback) automatique vers des règles d'affaires si le LLM est indisponible.

## ⚖️ 6. Comparatif Complet Avant vs. Après

| Dimension | ❌ Avant les Corrections (Prototype Initial) | ✅ Après les Améliorations (Moteur de Production) |
|---|---|---|
| **Construction de la Cible** | Cible statique (`1` = acheté une fois dans sa vie, `0` = jamais acheté). | **Cible Visit-Level dynamique** (`1` = commandé sur la facture $T$, `0` = skippé). |
| **Score ROC-AUC** | Faux score de 1.00 / 0.9999 par fuite de la variable `frequency`. | **Score réaliste de 0.8608** mesurant le vrai cycle de réapprovisionnement. |
| **Features** | 24 features brutes avec bruit et surapprentissage géologique (GPS). | **10 features cœur** épurées centrées sur la récence et le rythme d'achat. |
| **Découpage des Données** | Split aléatoire par client (fuite temporelle potentielle). | **Split Temporel** (Train $\le$ 2025 / Val Q1 2026 / Test Q2 2026+). |
| **Configuration XGBoost** | `max_depth=6`, `min_child_weight=1` (sensible au bruit). | `max_depth=5`, `min_child_weight=5` (haute régularisation anti-bruit). |
| **Classement des Recommandations** | Tri brut par probabilité ML. | **Business Re-Ranking** : $\text{final\_score} = \text{ml\_score} \times \text{timing\_boost} \times \text{trend\_boost}$. |
| **Dédoublonnage API** | Doublons d'articles possibles issus de factures historiques différentes. | **Dédoublonnage strict** conservant le snapshot produit le plus récent. |
| **Interface React (UI)** | Badges génériques et explications statiques affichées sur la carte. | **Modal d'Analyse LLM Llama 3.3** interactive avec fond flouté et cartes épurées. |

## 🧪 7. Cellule d'Inspection et Démonstration du Modèle

Exécutez la cellule ci-dessous pour charger les métadonnées et vérifier l'importance des variables du classifieur ré-entraîné.

In [ ]:
import os
import json
import joblib
import pandas as pd

model_path = '../src/models/classifier_lsat.joblib'
metadata_path = '../src/models/classifier_lsat_metadata.json'

if not os.path.exists(model_path):
    model_path = 'src/models/classifier_lsat.joblib'
    metadata_path = 'src/models/classifier_lsat_metadata.json'

if os.path.exists(model_path) and os.path.exists(metadata_path):
    model = joblib.load(model_path)
    with open(metadata_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    
    feature_cols = meta.get('feature_columns', [])
    importances = model.feature_importances_
    
    fi_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance (%)': [round(imp * 100, 2) for imp in importances]
    }).sort_values('Importance (%)', ascending=False).reset_index(drop=True)
    
    print("=== IMPORTANCE DES FEATURES DU CLASSIFIEUR (VISIT-LEVEL) ===")
    print(fi_df.to_string(index=False))
else:
    print('Artéfacts du modèle introuvables. Assurez-vous d\'avoir exécuté train_classifier.py.')